<a href="https://colab.research.google.com/github/op1154/MBA_Agents/blob/main/Memoria_Long_Term_salvando_emBD.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [17]:
!pip install -U langgraph -q
!pip install -U langgraph-checkpoint-sqlite -q
!pip install langchain-openai

In [18]:
!pip install langchain-openai

In [19]:
import uuid
import sqlite3
from langgraph.store.sqlite import SqliteStore
import os
from google.colab import userdata
# Defina sua chave de API da OpenAI aqui
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')

In [20]:


# 1. Configurar a conexão física com o arquivo SQLite no Colab
conn = sqlite3.connect("memoria_tutor.db", check_same_thread=False)

# 2. Inicializar o SqliteStore apontando para a conexão e a configuração de embedding
store = SqliteStore(conn, index={"embed": "openai:text-embedding-3-small", "dims": 1536})

# Executa as migrações/criações das tabelas internas do LangGraph no arquivo .db (Obrigatório para SQLite)
store.setup()

In [21]:
# =====================================================================

# Desfaz qualquer transação pendente que possa ter travado a conexão
try:
    conn.rollback()
except Exception:
    pass

# 1. Definindo o namespace que foi utilizado para salvar os dados
namespace = ("memories", "ana")

# 2. Recuperando todos os registros ou buscando por similaridade semântica usando search
resultados_busca = store.search(namespace, query="Qual o nome?", limit=5)
print("=== Resultados da Busca Inicial (search) ===")
for item in resultados_busca:
    print(f"ID: {item.key} | Conteúdo: {item.value}")

# Insere um dado passando (namespace, chave_única, dicionário_com_os_dados)
store.put(("memories", "ana"), "user_preference", {"data": "A Ana gosta de cafe sem açucar"})

# 1. Em vez de usar uma chave fixa como "user_preference", geramos chaves únicas (UUID)
chave_1 = str(uuid.uuid4())
chave_2 = str(uuid.uuid4())

# 2. Salvando múltiplas informações no mesmo namespace
store.put(("memories", "ana"), chave_1, {"data": "A Ana prefere estudar no período da manhã."})
store.put(("memories", "ana"), chave_2, {"data": "A Ana está estudando para a prova de Matemática."})

# 3. Agora, ao buscar, você verá todos os registros ativos neste namespace!
resultados = store.search(("memories", "ana"), query="Estudo", limit=10)
print(f"\n=== Total de registros encontrados: {len(resultados)} ===")
for item in resultados:
    print(f"Chave: {item.key} -> {item.value}")


=== Resultados da Busca Inicial (search) ===
ID: 9f97e773-9aea-4b22-b68f-e45698d9f02e | Conteúdo: {'data': 'A Ana está estudando para a prova de Matemática.'}

=== Total de registros encontrados: 4 ===
Chave: 4e4e8c05-4998-4065-a163-e334d9581687 -> {'data': 'A Ana está estudando para a prova de Matemática.'}
Chave: 9f97e773-9aea-4b22-b68f-e45698d9f02e -> {'data': 'A Ana está estudando para a prova de Matemática.'}
Chave: 61bb02ad-202f-4732-ba32-63a0d79ddd21 -> {'data': 'A Ana prefere estudar no período da manhã.'}
Chave: user_preference -> {'data': 'A Ana gosta de cafe sem açucar'}


In [27]:
# Busca semântica para saber o que a Ana gosta
query_busca = "Como a Ana gosta de café?"

resultados_gosto = store.search(("memories", "ana"), query=query_busca, limit=5)

print(f"=== Resultados da busca para: '{query_busca}' ===")
for item in resultados_gosto:
    print(f"Chave: {item.key} -> {item.value}")

=== Resultados da busca para: 'Como a Ana gosta de café?' ===
Chave: user_preference -> {'data': 'A Ana gosta de cafe sem açucar'}
Chave: 61bb02ad-202f-4732-ba32-63a0d79ddd21 -> {'data': 'A Ana prefere estudar no período da manhã.'}
Chave: 4e4e8c05-4998-4065-a163-e334d9581687 -> {'data': 'A Ana está estudando para a prova de Matemática.'}
Chave: 9f97e773-9aea-4b22-b68f-e45698d9f02e -> {'data': 'A Ana está estudando para a prova de Matemática.'}


In [29]:
# --- DELEÇÃO DA CHAVE DINÂMICA ---
# 1. Removemos a chave_1 gerada dinamicamente (atribuindo None ao valor)
store.put(("memories", "ana"), chave_1, None)

# 2. Tentamos buscar o registro removido diretamente usando a variável da chave
record = store.get(("memories", "ana"), chave_1)

# 3. Verificação de segurança para evitar o erro AttributeError
print("\n=== Verificação após deletar chave_1 ===")
if record is not None:
    print(record.value)
else:
    print(f"O registro para a chave '{chave_1}' foi deletado com sucesso no SQLite.")

# --- DELEÇÃO DA CHAVE FIXA ---
# 1. Para deletar um registro específico, passamos None no valor (value)
store.put(("memories", "ana"), "user_preference", None)

# 2. Vamos buscar novamente para comprovar que o registro 'user_preference' foi removido
resultados_restantes = store.search(("memories", "ana"), query="cafe", limit=5)
print("\n=== Registros restantes após a deleção de 'user_preference' ===")
if len(resultados_restantes) == 0:
    print("Nenhum registro encontrado sobre café (deletado com sucesso).")
for item in resultados_restantes:
    print(f"Chave: {item.key} -> {item.value}")


=== Verificação após deletar chave_1 ===
O registro para a chave '61bb02ad-202f-4732-ba32-63a0d79ddd21' foi deletado com sucesso no SQLite.

=== Registros restantes após a deleção de 'user_preference' ===
Chave: 4e4e8c05-4998-4065-a163-e334d9581687 -> {'data': 'A Ana está estudando para a prova de Matemática.'}
Chave: 9f97e773-9aea-4b22-b68f-e45698d9f02e -> {'data': 'A Ana está estudando para a prova de Matemática.'}
